# Data Exploration
**Project:** Multimodal detection of freezing of gait (FoG) in Parkinson's disease from video (skeleton + GNN) and IMU (LSTM)

**Dataset:** Public dataset of video, acceleration and angular velocity in individuals with Parkinson's disease during the turning-in-place task (Figshare, article 14984667)

**Goals of this notebook:**
1. Download the dataset and understand its structure
2. Match video and IMU files
3. Count freezing episodes and freezing time
4. Compare video and IMU durations
5. Check the labels against the clinical information file

## 1. Download the dataset
The data is downloaded directly from Figshare using the Figshare API.

In [36]:
# Ask the Figshare API for the list of files in the dataset (names, sizes, download links).
import requests

article_id = 14984667
response = requests.get(f"https://api.figshare.com/v2/articles/{article_id}")
files = response.json()["files"]

#for f in files:
    #print(f["name"], "|", round(f["size"] / 1e9, 2), "GB")

In [37]:
!mkdir -p /content/data

for f in files:
    name = f["name"]
    url = f["download_url"]
    !wget -c -q --show-progress -O "/content/data/{name}" "{url}"

/content/data/Video 100%[===================>]   4.67G  26.0MB/s    in 3m 9s   
/content/data/IMU.z 100%[===================>] 152.50M  18.9MB/s    in 8.7s    


In [38]:
# Unzip all downloaded zip files, then delete the zips to save disk space.
!cd /content/data && for z in *.zip; do unzip -q "$z" && rm "$z"; done
!ls -lh /content/data

replace IMU/SUB01_1.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
replace Videos/PDFE01_1.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
total 32K
drwxr-xr-x 2 root root  12K Oct  8 05:17 IMU
-rw-r--r-- 1 root root  13K Nov 21  2021 PDFEinfo.csv
drwxr-xr-x 2 root root 4.0K Oct  8 05:18 Videos


## 2. Dataset structure
The dataset contains three parts:
- `IMU/` – one file per trial (`SUBxx_y.csv`) and one standing recording per subject (`SUBxx_standing.csv`)
- `Videos/` – one video per trial (`PDFExx_y.mp4`)
- `PDFEinfo.csv` – demographic and clinical information per patient

In [39]:
# List every folder and file inside the dataset folder.
from pathlib import Path

DATA_DIR= Path('/content/data')
[str(item) for item in DATA_DIR.rglob("*")]

['/content/data/Videos',
 '/content/data/PDFEinfo.csv',
 '/content/data/IMU',
 '/content/data/Videos/PDFE35_2.mp4',
 '/content/data/Videos/PDFE18_3.mp4',
 '/content/data/Videos/PDFE09_1.mp4',
 '/content/data/Videos/PDFE26_2.mp4',
 '/content/data/Videos/PDFE13_1.mp4',
 '/content/data/Videos/PDFE04_1.mp4',
 '/content/data/Videos/PDFE20_1.mp4',
 '/content/data/Videos/PDFE21_3.mp4',
 '/content/data/Videos/PDFE05_1.mp4',
 '/content/data/Videos/PDFE23_1.mp4',
 '/content/data/Videos/PDFE27_1.mp4',
 '/content/data/Videos/PDFE15_1.mp4',
 '/content/data/Videos/PDFE17_1.mp4',
 '/content/data/Videos/PDFE29_1.mp4',
 '/content/data/Videos/PDFE23_2.mp4',
 '/content/data/Videos/PDFE06_2.mp4',
 '/content/data/Videos/PDFE21_2.mp4',
 '/content/data/Videos/PDFE31_3.mp4',
 '/content/data/Videos/PDFE26_1.mp4',
 '/content/data/Videos/PDFE25_2.mp4',
 '/content/data/Videos/PDFE27_3.mp4',
 '/content/data/Videos/PDFE02_1.mp4',
 '/content/data/Videos/PDFE05_2.mp4',
 '/content/data/Videos/PDFE07_3.mp4',
 '/content

In [40]:
# Count the trial files only: names ending with "_<number>".
# This excludes the "standing" IMU files and the PDFEinfo.csv file.
import os
import re

video_Counter = 0
IMU_Counter = 0


for root, dirs, files in os.walk('/content/data/IMU'):
    for file in files:
        if re.search(r"_[0-9]\.csv$", file):
           IMU_Counter += 1


for root, dirs, files in os.walk('/content/data/Videos'):
    for file in files:
        if re.search(r"_[0-9]\.mp4$", file):
           video_Counter += 1


print ("there are",video_Counter,"videos and",IMU_Counter,"IMU files in this dataset")

there are 73 videos and 71 IMU files in this dataset


## 3. Matching video and IMU trials
Videos are named `PDFExx_y` and IMU files `SUBxx_y` (xx = subject, y = trial).
To compare them, the prefixes are removed, and the two lists of names are compared with sets.

In [41]:
# Collect the names of all IMU trial files (without extension), sorted.
IMU_path= Path(DATA_DIR/"IMU")
IMU_files=[]

for file in IMU_path.iterdir():
    if re.search(r"_[0-9]\.csv$",file.name):
       file_name= Path(file.name).stem
       IMU_files.append (file_name)

IMU_files = sorted(IMU_files)
print (IMU_files)

['SUB01_1', 'SUB01_2', 'SUB02_1', 'SUB03_1', 'SUB03_2', 'SUB03_3', 'SUB04_1', 'SUB05_1', 'SUB05_2', 'SUB06_1', 'SUB06_2', 'SUB07_1', 'SUB07_2', 'SUB07_3', 'SUB08_1', 'SUB09_1', 'SUB09_2', 'SUB10_1', 'SUB10_2', 'SUB11_1', 'SUB11_2', 'SUB12_1', 'SUB13_1', 'SUB14_1', 'SUB14_2', 'SUB14_3', 'SUB15_1', 'SUB16_1', 'SUB16_2', 'SUB17_1', 'SUB18_1', 'SUB18_2', 'SUB18_3', 'SUB19_1', 'SUB19_2', 'SUB20_1', 'SUB20_2', 'SUB20_3', 'SUB21_1', 'SUB21_2', 'SUB21_3', 'SUB22_1', 'SUB23_1', 'SUB23_2', 'SUB24_1', 'SUB25_1', 'SUB25_2', 'SUB25_3', 'SUB26_1', 'SUB26_2', 'SUB26_3', 'SUB27_1', 'SUB27_2', 'SUB27_3', 'SUB28_1', 'SUB28_2', 'SUB28_3', 'SUB29_1', 'SUB30_1', 'SUB30_2', 'SUB31_1', 'SUB31_2', 'SUB31_3', 'SUB32_1', 'SUB32_2', 'SUB32_3', 'SUB33_1', 'SUB34_1', 'SUB35_1', 'SUB35_2', 'SUB35_3']


In [42]:
# Remove the "SUB" prefix so IMU names can be compared with video names.
IMU_files = [name.replace("SUB", "") for name in IMU_files]
print (IMU_files)

['01_1', '01_2', '02_1', '03_1', '03_2', '03_3', '04_1', '05_1', '05_2', '06_1', '06_2', '07_1', '07_2', '07_3', '08_1', '09_1', '09_2', '10_1', '10_2', '11_1', '11_2', '12_1', '13_1', '14_1', '14_2', '14_3', '15_1', '16_1', '16_2', '17_1', '18_1', '18_2', '18_3', '19_1', '19_2', '20_1', '20_2', '20_3', '21_1', '21_2', '21_3', '22_1', '23_1', '23_2', '24_1', '25_1', '25_2', '25_3', '26_1', '26_2', '26_3', '27_1', '27_2', '27_3', '28_1', '28_2', '28_3', '29_1', '30_1', '30_2', '31_1', '31_2', '31_3', '32_1', '32_2', '32_3', '33_1', '34_1', '35_1', '35_2', '35_3']


In [43]:
# Collect the names of all video trial files.
video_path= Path(DATA_DIR/"Videos")
video_files=[]

for file in video_path.iterdir():
    if re.search(r"_[0-9]\.mp4$",file.name):
       file_name= Path(file.name).stem
       video_files.append (file_name)

video_files = sorted(video_files)
print (video_files)

['PDFE01_1', 'PDFE01_2', 'PDFE02_1', 'PDFE03_1', 'PDFE03_2', 'PDFE03_3', 'PDFE04_1', 'PDFE05_1', 'PDFE05_2', 'PDFE06_1', 'PDFE06_2', 'PDFE07_1', 'PDFE07_2', 'PDFE07_3', 'PDFE08_1', 'PDFE09_1', 'PDFE09_2', 'PDFE10_1', 'PDFE10_2', 'PDFE10_3', 'PDFE11_1', 'PDFE11_2', 'PDFE12_1', 'PDFE13_1', 'PDFE14_1', 'PDFE14_2', 'PDFE14_3', 'PDFE15_1', 'PDFE16_1', 'PDFE16_2', 'PDFE17_1', 'PDFE18_1', 'PDFE18_2', 'PDFE18_3', 'PDFE19_1', 'PDFE19_2', 'PDFE20_1', 'PDFE20_2', 'PDFE20_3', 'PDFE21_1', 'PDFE21_2', 'PDFE21_3', 'PDFE22_1', 'PDFE23_1', 'PDFE23_2', 'PDFE24_1', 'PDFE25_1', 'PDFE25_2', 'PDFE25_3', 'PDFE26_1', 'PDFE26_2', 'PDFE26_3', 'PDFE27_1', 'PDFE27_2', 'PDFE27_3', 'PDFE28_1', 'PDFE28_2', 'PDFE28_3', 'PDFE29_1', 'PDFE30_1', 'PDFE30_2', 'PDFE30_3', 'PDFE31_1', 'PDFE31_2', 'PDFE31_3', 'PDFE32_1', 'PDFE32_2', 'PDFE32_3', 'PDFE33_1', 'PDFE34_1', 'PDFE35_1', 'PDFE35_2', 'PDFE35_3']


In [44]:
# Remove the "PDFE" prefix so video names can be compared with IMU names.
video_files = [name.replace("PDFE", "") for name in video_files]
print (video_files)

['01_1', '01_2', '02_1', '03_1', '03_2', '03_3', '04_1', '05_1', '05_2', '06_1', '06_2', '07_1', '07_2', '07_3', '08_1', '09_1', '09_2', '10_1', '10_2', '10_3', '11_1', '11_2', '12_1', '13_1', '14_1', '14_2', '14_3', '15_1', '16_1', '16_2', '17_1', '18_1', '18_2', '18_3', '19_1', '19_2', '20_1', '20_2', '20_3', '21_1', '21_2', '21_3', '22_1', '23_1', '23_2', '24_1', '25_1', '25_2', '25_3', '26_1', '26_2', '26_3', '27_1', '27_2', '27_3', '28_1', '28_2', '28_3', '29_1', '30_1', '30_2', '30_3', '31_1', '31_2', '31_3', '32_1', '32_2', '32_3', '33_1', '34_1', '35_1', '35_2', '35_3']


In [45]:
# Find trials that exist in only one modality.
set_video = set(video_files)
set_IMU = set(IMU_files)

only_in_video = sorted(set_video - set_IMU)
only_in_IMU = sorted(set_IMU - set_video)

print("Only in list_video:", only_in_video)
print("Only in list_IMU:", only_in_IMU)

Only in list_video: ['10_3', '30_3']
Only in list_IMU: []


**Result:** 73 videos and 71 IMU trials. Two videos have no IMU file: **PDFE10_3** and **PDFE30_3**.
**Decision:** these 2 trials are excluded. The project uses the **71 trials** that have both video and IMU.

## 4. Structure of the IMU files
**Important:** the IMU files are in **Excel format despite the `.csv` extension**, so they must be read with `pd.read_excel` (`pd.read_csv` fails).

Each file has 15,360 rows × 9 columns. At 128 Hz, this equals **120 seconds**.
Columns:
- `ACC ML / AP / SI` – acceleration (g) in 3 directions: ML = side-to-side, AP = front-back, SI = up-down
- `GYR ML / AP / SI` – angular velocity (deg/s) around the same 3 directions
- `Freezing event [flag]` – the label: 0 = no freezing, 1 = freezing

In [46]:
# Load one IMU file to inspect its shape and columns.
import pandas as pd

df = pd.read_excel('/content/data/IMU/SUB01_1.csv')
print(df.shape)
df.head(5)

(15360, 9)


,Frame #,Time [s],ACC ML [g],ACC AP [g],ACC SI [g],GYR ML [deg/s],GYR AP [deg/s],GYR SI [deg/s],Freezing event [flag]
0,1,0.007812,0.191745,-0.511756,1.014700,-10.448125,58.094807,29.161523,0
1,2,0.015625,0.226322,-0.258745,1.084140,-1.247650,56.511776,35.680386,0
2,3,0.023438,0.405319,-0.561918,0.915292,-3.488858,49.336805,38.986192,0
3,4,0.031250,0.620089,-0.285977,0.975270,7.025583,39.408962,71.966275,0
4,5,0.039062,0.488221,-1.063820,0.640557,-1.527005,33.880341,79.829072,0


In [47]:
# Load all 71 IMU trial files into a dictionary: key = file name, value = DataFrame.
IMU_dictionary = {}

for file in IMU_path.iterdir():
    if re.fullmatch(r"SUB\d+_\d+\.csv", file.name):
       df = pd.read_excel(file)
       IMU_dictionary[file.stem] = df

IMU_dictionary.keys()

dict_keys(['SUB14_1', 'SUB27_2', 'SUB17_1', 'SUB31_2', 'SUB07_1', 'SUB16_2', 'SUB15_1', 'SUB09_1', 'SUB27_3', 'SUB31_3', 'SUB11_1', 'SUB35_2', 'SUB09_2', 'SUB16_1', 'SUB13_1', 'SUB29_1', 'SUB33_1', 'SUB31_1', 'SUB28_2', 'SUB26_2', 'SUB01_2', 'SUB20_3', 'SUB14_2', 'SUB27_1', 'SUB19_2', 'SUB03_1', 'SUB05_1', 'SUB01_1', 'SUB03_3', 'SUB26_3', 'SUB22_1', 'SUB12_1', 'SUB06_2', 'SUB35_3', 'SUB26_1', 'SUB32_1', 'SUB06_1', 'SUB24_1', 'SUB07_3', 'SUB30_1', 'SUB25_2', 'SUB07_2', 'SUB23_2', 'SUB18_1', 'SUB30_2', 'SUB35_1', 'SUB02_1', 'SUB21_1', 'SUB25_1', 'SUB25_3', 'SUB32_2', 'SUB34_1', 'SUB04_1', 'SUB19_1', 'SUB28_1', 'SUB11_2', 'SUB20_2', 'SUB14_3', 'SUB03_2', 'SUB10_1', 'SUB21_3', 'SUB05_2', 'SUB23_1', 'SUB18_2', 'SUB18_3', 'SUB08_1', 'SUB20_1', 'SUB10_2', 'SUB28_3', 'SUB21_2', 'SUB32_3'])

In [48]:
IMU_dictionary['SUB20_2']

,Frame #,Time [s],ACC ML [g],ACC AP [g],ACC SI [g],GYR ML [deg/s],GYR AP [deg/s],GYR SI [deg/s],Freezing event [flag]
0,1,0.007812,0.193571,1.256703,1.776516,15.947759,-41.021053,-67.557149,0
1,2,0.015625,0.203467,0.655085,2.000127,14.501161,-14.553426,-123.976626,0
2,3,0.023438,0.122778,-1.249579,2.000827,37.044831,21.538498,-224.017121,0
3,4,0.031250,-1.134741,-1.440374,1.106971,54.032249,15.759879,-115.617024,0
4,5,0.039062,-1.235796,-0.402316,0.538383,22.665166,-20.655016,-29.414845,0
...,...,...,...,...,...,...,...,...,...
15355,15356,119.968750,0.434834,-0.467927,1.151717,15.955942,-7.784984,-62.398054,0
15356,15357,119.976562,0.354891,-0.487646,1.033820,13.475365,-8.111375,-47.341805,0
15357,15358,119.984375,0.258420,-0.532852,0.874045,15.076274,-13.005600,-27.292882,0
15358,15359,119.992188,0.328958,-0.778827,0.771600,20.287951,-16.365306,1.923112,0


## 5. Freezing episodes and freezing time
An **episode** is a continuous run of 1s in the flag column. It starts where the flag changes from 0 to 1 (or at the first row, if the file starts with 1).
Freezing time (s) = number of rows with flag 1 / 128.

In [49]:
# For each trial: count freezing episodes, freezing time, and total recording time.
# Results are collected in a table (one row per trial).
df_freezing=[]
for name, df_n in IMU_dictionary.items():
    new_name=name.removeprefix('SUB')
    split_name=new_name.split("_")
    subject=int(split_name[0])
    trial=int(split_name[1])
    n_episode=0
    if df_n.iloc[0]["Freezing event [flag]"]==1:
       n_episode+=1
    for i in range(len(df_n) - 1):
        if df_n.iloc[i]["Freezing event [flag]"]==0 and df_n.iloc[i+1]["Freezing event [flag]"]==1:
           n_episode+=1

    freezing_rows = (df_n["Freezing event [flag]"] == 1).sum()
    freezing_seconds=freezing_rows/128
    total_time_IMU=len(df_n)/128
    df_freezing.append({"File_Name":name,"Subject":subject,"Trial":trial,"Number_Episode":n_episode,"Freezing_Seconds":freezing_seconds,"Total_Time_IMU":total_time_IMU})

df_freezing=pd.DataFrame(df_freezing)
df_freezing = df_freezing.sort_values("Subject").reset_index(drop=True)



In [50]:
df_freezing

,File_Name,Subject,Trial,Number_Episode,Freezing_Seconds,Total_Time_IMU
0,SUB01_1,1,1,4,115.101562,120.0
1,SUB01_2,1,2,2,115.960938,120.0
2,SUB02_1,2,1,4,107.585938,120.0
3,SUB03_3,3,3,1,1.437500,120.0
4,SUB03_2,3,2,0,0.000000,120.0
...,...,...,...,...,...,...
66,SUB33_1,33,1,6,42.351562,120.0
67,SUB34_1,34,1,3,25.195312,120.0
68,SUB35_3,35,3,0,0.000000,120.0
69,SUB35_1,35,1,0,0.000000,120.0


In [51]:
# Dataset summary: total episodes, % of time freezing, subjects with at least one episode.
# Subjects are counted per subject, not per trial (a subject can have up to 3 trials).
total_episode=df_freezing["Number_Episode"].sum()
freezing_percentage=df_freezing["Freezing_Seconds"].sum()/df_freezing["Total_Time_IMU"].sum()*100

subject_with_episode=0
for i in range (1,36):
    rows_of_subject = df_freezing[df_freezing["Subject"] == i]
    episode_per_subject=rows_of_subject["Number_Episode"].sum()
    if episode_per_subject>0:
       subject_with_episode+=1


print("total number of episode:",total_episode)
print(f"percentage of all time that is freezing:{freezing_percentage:.1f}%")
print("number of subjects with at least one episode:",subject_with_episode)
print(f"total freezing time (s): {df_freezing['Freezing_Seconds'].sum():.0f}")

total number of episode: 173
percentage of all time that is freezing:18.9%
number of subjects with at least one episode: 26
total freezing time (s): 1612


**Results:**
- **173 freezing episodes** in total (matches the number reported in the literature for this dataset)
- **1,612 s of freezing = 18.9%** of all recorded time (literature: 1,611 s)
- **26 of 35 subjects** have at least one episode
- Patients differ strongly: e.g. SUB01_1 freezes 115 of 120 s, while SUB35 never freezes. This must be considered when splitting patients into training and test sets.

## 6. Video properties and comparison with IMU duration
Video and IMU must cover the same time period, otherwise the freezing labels will not match the video frames.

In [52]:
# Read FPS, frame count, and duration of one video, then release it.
import cv2

video = cv2.VideoCapture('/content/data/Videos/PDFE04_1.mp4')

if not video.isOpened():
    print("Error: could not open the video. Check the file name and path.")
else:
    fps      = video.get(cv2.CAP_PROP_FPS)
    n_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = n_frames / fps

    print("FPS:", fps)
    print("Frames:", n_frames)
    print(f"Duration (seconds): {duration:.1f}")

video.release()

FPS: 29.978873295485865
Frames: 3390
Duration (seconds): 113.1


In [53]:
# Open all trial videos and store them in a dictionary: key = file name, value = VideoCapture.
video_dictionary = {}

for file in video_path.iterdir():
    if re.fullmatch(r"PDFE\d+_\d+\.mp4", file.name):
       v_file = cv2.VideoCapture(file)
       video_dictionary[file.stem] = v_file
video_dictionary.keys()

dict_keys(['PDFE35_2', 'PDFE18_3', 'PDFE09_1', 'PDFE26_2', 'PDFE13_1', 'PDFE04_1', 'PDFE20_1', 'PDFE21_3', 'PDFE05_1', 'PDFE23_1', 'PDFE27_1', 'PDFE15_1', 'PDFE17_1', 'PDFE29_1', 'PDFE23_2', 'PDFE06_2', 'PDFE21_2', 'PDFE31_3', 'PDFE26_1', 'PDFE25_2', 'PDFE27_3', 'PDFE02_1', 'PDFE05_2', 'PDFE07_3', 'PDFE18_1', 'PDFE11_1', 'PDFE32_1', 'PDFE07_2', 'PDFE03_3', 'PDFE10_2', 'PDFE31_2', 'PDFE33_1', 'PDFE25_3', 'PDFE32_3', 'PDFE14_3', 'PDFE08_1', 'PDFE10_3', 'PDFE24_1', 'PDFE31_1', 'PDFE27_2', 'PDFE07_1', 'PDFE32_2', 'PDFE20_2', 'PDFE28_1', 'PDFE30_3', 'PDFE22_1', 'PDFE01_2', 'PDFE14_1', 'PDFE21_1', 'PDFE14_2', 'PDFE06_1', 'PDFE26_3', 'PDFE19_1', 'PDFE30_2', 'PDFE30_1', 'PDFE28_2', 'PDFE03_1', 'PDFE16_1', 'PDFE35_3', 'PDFE09_2', 'PDFE28_3', 'PDFE01_1', 'PDFE12_1', 'PDFE25_1', 'PDFE11_2', 'PDFE18_2', 'PDFE03_2', 'PDFE10_1', 'PDFE19_2', 'PDFE34_1', 'PDFE35_1', 'PDFE20_3', 'PDFE16_2'])

In [54]:
# Build a table with each video's FPS, number of frames, and duration (s).
df_video=[]
for name, v_file in video_dictionary.items():
    new_name=name.removeprefix('PDFE')
    split_name=new_name.split("_")
    subject=int(split_name[0])
    trial=int(split_name[1])

    frame_per_seconde= v_file.get(cv2.CAP_PROP_FPS)
    number_frames = int(v_file.get(cv2.CAP_PROP_FRAME_COUNT))
    video_duration = number_frames / frame_per_seconde
    df_video.append({"File_Name":name,"Subject":subject,"Trial":trial,"Frames_Per_Seconds":frame_per_seconde,"Number_Frame":number_frames,"Video_Duration":video_duration})

df_video=pd.DataFrame(df_video)
df_video = df_video.sort_values("Subject").reset_index(drop=True)



In [55]:
df_video

,File_Name,Subject,Trial,Frames_Per_Seconds,Number_Frame,Video_Duration
0,PDFE01_1,1,1,29.978362,3598,120.019900
1,PDFE01_2,1,2,29.978187,3675,122.589133
2,PDFE02_1,2,1,29.978159,3688,123.022900
3,PDFE03_3,3,3,29.978284,3632,121.154367
4,PDFE03_1,3,1,29.978286,3631,121.121000
...,...,...,...,...,...,...
68,PDFE33_1,33,1,29.978309,3621,120.787333
69,PDFE34_1,34,1,29.978323,3615,120.587133
70,PDFE35_3,35,3,29.978302,3624,120.887433
71,PDFE35_2,35,2,29.978343,3606,120.286833


In [56]:
# Merge the video table with the IMU table (keeps only the 71 trials present in both).
# Difference = video duration - IMU duration (s).
df_freezing=df_freezing.drop(columns="File_Name")
df_video=df_video.drop(columns="File_Name")
df_V_IMU=pd.merge(df_video, df_freezing, on=["Subject", "Trial"])
df_V_IMU["Difference"] = df_V_IMU["Video_Duration"] - df_V_IMU["Total_Time_IMU"]

In [57]:
df_V_IMU

,Subject,Trial,Frames_Per_Seconds,Number_Frame,Video_Duration,Number_Episode,Freezing_Seconds,Total_Time_IMU,Difference
0,1,1,29.978362,3598,120.019900,4,115.101562,120.0,0.019900
1,1,2,29.978187,3675,122.589133,2,115.960938,120.0,2.589133
2,2,1,29.978159,3688,123.022900,4,107.585938,120.0,3.022900
3,3,3,29.978284,3632,121.154367,1,1.437500,120.0,1.154367
4,3,1,29.978286,3631,121.121000,1,1.265625,120.0,1.121000
...,...,...,...,...,...,...,...,...,...
66,33,1,29.978309,3621,120.787333,6,42.351562,120.0,0.787333
67,34,1,29.978323,3615,120.587133,3,25.195312,120.0,0.587133
68,35,3,29.978302,3624,120.887433,0,0.000000,120.0,0.887433
69,35,2,29.978343,3606,120.286833,0,0.000000,120.0,0.286833


In [58]:
diff_min=df_V_IMU["Difference"].min()
diff_max=df_V_IMU["Difference"].max()
diff_mean=df_V_IMU["Difference"].mean()

print(f"Minimum difference between video duration and total time in IMU is:{diff_min:.1f}")
print(f"Maximum difference between video duration and total time in IMU is:{diff_max:.1f}")
print(f"mean of difference between video duration and total time in IMU is:{diff_mean:.1f}")

Minimum difference between video duration and total time in IMU is:-7.9
Maximum difference between video duration and total time in IMU is:3.0
mean of difference between video duration and total time in IMU is:0.2


In [59]:
# List the trials where the video is shorter than the IMU recording.
df_V_IMU[df_V_IMU["Video_Duration"] < df_V_IMU["Total_Time_IMU"]]

,Subject,Trial,Frames_Per_Seconds,Number_Frame,Video_Duration,Number_Episode,Freezing_Seconds,Total_Time_IMU,Difference
6,4,1,29.978873,3390,113.079633,8,17.023438,120.0,-6.920367
15,9,1,29.978952,3360,112.078633,0,0.000000,120.0,-7.921367
16,9,2,29.978719,3450,115.081633,4,20.851562,120.0,-4.918367
39,21,1,29.978364,3597,119.986533,0,0.000000,120.0,-0.013467
47,25,1,29.978364,3597,119.986533,0,0.000000,120.0,-0.013467
50,26,1,29.978364,3597,119.986533,0,0.000000,120.0,-0.013467
57,29,1,29.978755,3436,114.614500,11,47.195312,120.0,-5.385500


**Results:** videos are recorded at ~30 fps (29.978). Video duration − IMU duration ranges from **−7.9 s to +3.0 s** (mean 0.2 s):
- **64 videos:** 0 to 3 s longer than the IMU
- **3 videos (21_1, 25_1, 26_1):** −0.013 s, smaller than half a frame, so effectively equal
- **4 videos clearly shorter:** 04_1 (−6.9 s), 09_1 (−7.9 s), 09_2 (−4.9 s), 29_1 (−5.4 s); three of them contain freezing episodes

**Conclusion:** the offset between video and IMU differs for each trial.

## 7. Clinical information file (PDFEinfo.csv)
This file is a real CSV, but in European Excel style: separator `;`, encoding `latin-1`, and `-` for missing values.
It has one row per patient (35 patients + empty rows) and 71 columns:
- demographics (age, weight, height, gender, disease duration, more affected side, L-Dopa dose)
- clinical scales per session (e.g. UPDRS-III, H&Y, NFoG-Q, MiniBESTest, TUG)
- freezing results per session (episode start–end times, total time, number of episodes)
- cognitive tests (MoCA, Stroop, etc.)

⚠️ The freezing columns contain the labels, so they must **never** be used as model input (label leakage).
They are used here only to verify the IMU labels, and later for subgroup analysis (e.g. by disease severity).

In [60]:
# Load the clinical file with the correct separator, encoding, and missing-value symbol.
# Clean the column names (some contain line breaks).
info = pd.read_csv(DATA_DIR / "PDFEinfo.csv",
                   sep=";",
                   encoding="latin-1",
                   na_values="-")
info.columns = info.columns.str.replace("\n", " ").str.strip()
info = info.dropna(subset=["ID"])

In [61]:
# Create a "Subject" column from the patient ID (e.g. "PDFE03" -> "03").
info.insert(0, "Subject", info["ID"].str.removeprefix("PDFE"))
info = info.drop(columns=["ID"])

In [62]:
# Reshape the clinical episode counts: from one column per session to one row per subject and trial.
New_info = []

for t in [1, 2, 3]:
    episode_columns = f"Session {t} - numbers of FoG episodes (n)"
    info_short = info[["Subject", episode_columns]].copy()
    info_short = info_short.rename(columns={episode_columns: "Number_Episode"})
    info_short["Trial"] = t
    New_info.append(info_short)

n_episode_info = pd.concat(New_info)
n_episode_info = n_episode_info.dropna(subset=["Number_Episode"])
n_episode_info = n_episode_info.sort_values(["Subject", "Trial"]).reset_index(drop=True)
n_episode_info = n_episode_info[["Subject", "Trial", "Number_Episode"]]
n_episode_info["Subject"] = n_episode_info["Subject"].astype(int)
n_episode_info.head()

,Subject,Trial,Number_Episode
0,1,1,4.0
1,1,2,2.0
2,2,1,4.0
3,3,1,1.0
4,3,2,0.0


In [63]:
# Put the IMU episode counts and the clinical episode counts side by side.
info_IMU = pd.merge(df_V_IMU[["Subject", "Trial", "Number_Episode"]],
                    n_episode_info,
                    on=["Subject", "Trial"],
                    suffixes=("_IMU", "_info"))

In [64]:
# List the trials where the two episode counts differ (an empty table means full agreement).
info_IMU[info_IMU["Number_Episode_IMU"] != info_IMU["Number_Episode_info"]]

,Subject,Trial,Number_Episode_IMU,Number_Episode_info


**Result:** the episode counts agree for **all 71 trials** (0 differences).
This confirms that:
1. The IMU labels are consistent with the clinical records
2. "Session N" in PDFEinfo corresponds to trial N in the file names

## Summary
- 35 patients, **71 trials** with both video and IMU (excluded: PDFE10_3, PDFE30_3)
- IMU: 128 Hz, 120 s per trial, 6 channels, one shank sensor; files in Excel format despite `.csv`
- Video: ~30 fps, durations 112–123 s; the offset to the IMU varies per trial → alignment needed
- **173 freezing episodes**, **1,612 s of freezing (18.9% of time)**, 26 of 35 patients freeze
- IMU labels agree 100% with the clinical information file
